# Day 3 Session 1 -- RAG Demos

These demos walk through the core retrieval-augmented generation building blocks you will use as an engineer building McKinsey's consulting knowledge platform. Users (consultants) query the system; your job is to make retrieval fast, accurate, and scalable.

**Demos in this notebook:**
1. Embedding Models -- creating embeddings, computing cosine similarity, semantic search
2. Vector Stores -- indexing and querying documents with ChromaDB
3. Chunking -- section-aware splitting for structured consulting reports

In [1]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters chromadb openai python-dotenv numpy


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


---
## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import numpy as np
import json

print("All imports successful!")

All imports successful!


---
## Demo 1: Embedding Models -- Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In a consulting context, embeddings let the platform find relevant strategy insights, industry analyses, and engagement playbooks even when the exact terminology differs -- e.g., a user query about "post-merger synergies" should retrieve documents discussing "integration value capture."

In [3]:
# Demo 1 - Embedding Models (McKinsey Consulting Knowledge)

client = openai.OpenAI()

# Step 1: Create embeddings for McKinsey consulting knowledge snippets
texts = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and cultural alignment between merging entities.",
    "Omnichannel retail strategy must prioritize seamless customer experience across physical and digital touchpoints.",
    "Supply chain resilience requires diversification of sourcing and near-shoring of critical components.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")

# Step 2: Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nSimilarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

# Step 3: Semantic search -- consulting query
query = "What are best practices for post-merger integration?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

print(f"\nQuery: {query}")
print("Ranked results:")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

Number of embeddings: 5
Embedding dimensions: 1536
First 5 values of embedding 0: [0.005741119384765625, 0.020233154296875, 0.07269287109375, -0.001773834228515625, 0.0177459716796875]

Similarity matrix:
                                               [0]  [1]  [2]  [3]  [4]
[0]   Digital transformation in financial serv 1.00 0.30 0.37 0.28 0.03
[1]   Post-merger integration success depends  0.30 1.00 0.31 0.26 0.04
[2]   Omnichannel retail strategy must priorit 0.37 0.31 1.00 0.32 -0.04
[3]   Supply chain resilience requires diversi 0.28 0.26 0.32 1.00 -0.05
[4]       The weather today is sunny and warm. 0.03 0.04 -0.04 -0.05 1.00

Query: What are best practices for post-merger integration?
Ranked results:
  0.6188 | Post-merger integration success depends on Day-1 readiness and cultural alignment between merging entities.
  0.3053 | Omnichannel retail strategy must prioritize seamless customer experience across physical and digital touchpoints.
  0.2733 | Digital transformation in fi

---
## Demo 2: Vector Stores -- Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

Here we build a **consulting knowledge base** -- indexing insights across strategy, operations, M&A, digital transformation, and organizational design so users can quickly surface relevant prior work.

In [4]:
# Demo 2 - Vector Stores with ChromaDB (McKinsey Knowledge Base)

# Step 1: Create a ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="mckinsey_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# Step 2: Add McKinsey consulting knowledge documents with metadata
documents = [
    "Digital transformation in financial services requires a phased approach: digitize core processes, then build new digital offerings, and finally reimagine the business model.",
    "Post-merger integration should follow a 100-day plan covering Day-1 readiness, synergy capture, cultural integration, and operating model design.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across all channels to deliver a seamless experience.",
    "Private equity value creation levers include revenue growth acceleration, operational efficiency, strategic M&A, and balance sheet optimization.",
    "Organizational restructuring requires a clear target operating model, role clarity, and a change management program to drive adoption.",
    "ESG strategy should be embedded into core business operations, not treated as a standalone compliance exercise, to create long-term shareholder value.",
    "Healthcare cost transformation requires addressing clinical variation, supply chain optimization, and administrative simplification simultaneously.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, cultural, and operational integration risks."
]

# Use OpenAI embeddings via the API
client = openai.OpenAI()
emb_response = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=documents)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[
        {"source": "digital_transformation_playbook", "practice": "Digital"},
        {"source": "ma_integration_guide", "practice": "M&A"},
        {"source": "retail_strategy_whitepaper", "practice": "Retail"},
        {"source": "pe_value_creation_framework", "practice": "Private Equity"},
        {"source": "org_design_handbook", "practice": "Organization"},
        {"source": "esg_strategy_report", "practice": "Sustainability"},
        {"source": "healthcare_cost_study", "practice": "Healthcare"},
        {"source": "cross_border_ma_guide", "practice": "M&A"},
    ]
)
print(f"Indexed {collection.count()} documents in McKinsey knowledge base")

# Step 3: Query the collection -- a typical consultant research question
query = "How should a retailer approach omnichannel transformation?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"\nQuery: {query}")
print("Top 3 results:")
for i, (doc, distance, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta['source']} | {meta['practice']}] (dist={distance:.4f}) {doc}")

Indexed 8 documents in McKinsey knowledge base

Query: How should a retailer approach omnichannel transformation?
Top 3 results:
  1. [retail_strategy_whitepaper | Retail] (dist=0.3626) Omnichannel retail strategy must unify inventory, pricing, and customer data across all channels to deliver a seamless experience.
  2. [digital_transformation_playbook | Digital] (dist=0.4777) Digital transformation in financial services requires a phased approach: digitize core processes, then build new digital offerings, and finally reimagine the business model.
  3. [org_design_handbook | Organization] (dist=0.5740) Organizational restructuring requires a clear target operating model, role clarity, and a change management program to drive adoption.


---
## Demo 3: Section-Aware Chunking for Consulting Reports

How you split documents directly affects retrieval quality. McKinsey documents typically follow a structured format: **Executive Summary, Key Findings, Detailed Analysis, Recommendations, and Appendices**. Section-aware splitting preserves this logical structure by prioritizing section boundaries as split points.

In [5]:
# Demo 3 - Section-Aware Chunking (McKinsey Consulting Report)

sample_doc = """# Post-Merger Integration: A Best Practice Guide

Post-merger integration (PMI) is the critical phase that determines whether an M&A transaction delivers its intended value. McKinsey research shows that 70% of mergers fail to achieve projected synergies, most often due to integration execution failures.

## Executive Summary

Successful post-merger integration requires a structured 100-day plan, early synergy identification, cultural alignment, and disciplined execution. Our analysis of 200+ transactions reveals that Day-1 readiness is the single strongest predictor of integration success.

## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:

1. Leadership alignment: Joint leadership teams must be appointed within the first two weeks, with clear decision rights and accountability.
2. Synergy capture: Revenue and cost synergies must be quantified with bottom-up rigor and tracked through a dedicated Integration Management Office (IMO).
3. Cultural integration: Cultural differences must be proactively addressed through joint team workshops, shared values articulation, and visible leadership behavior.

## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) focuses on stabilization and quick wins. Phase 2 (Days 31-100) addresses operating model design and synergy capture. Phase 3 (Months 4-12) drives full integration and transformation.

## Appendix

Detailed case studies from recent financial services and healthcare M&A transactions are provided in the supplementary materials, including synergy tracking templates and cultural assessment frameworks."""

# Section-aware splitting (respects consulting report structure)
section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
section_chunks = section_splitter.split_text(sample_doc)

print(f"Section-aware splitting: {len(section_chunks)} chunks")
for i, chunk in enumerate(section_chunks):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:90]}...")

# Compare with naive fixed-size splitting
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")), separators=["\n\n", "\n", ". ", " "]
)
naive_chunks = naive_splitter.split_text(sample_doc)

print(f"\nNaive fixed-size splitting: {len(naive_chunks)} chunks")
for i, chunk in enumerate(naive_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:90]}...")

print(f"\nSection-aware preserves document structure by splitting on ## headers first.")
print(f"Comparison: Section-aware={len(section_chunks)} chunks vs Naive={len(naive_chunks)} chunks")

Section-aware splitting: 8 chunks
  Chunk 1 (48 chars): # Post-Merger Integration: A Best Practice Guide...
  Chunk 2 (254 chars): Post-merger integration (PMI) is the critical phase that determines whether an M&A transac...
  Chunk 3 (290 chars): ## Executive Summary

Successful post-merger integration requires a structured 100-day pla...
  Chunk 4 (90 chars): ## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:...
  Chunk 5 (296 chars): 1. Leadership alignment: Joint leadership teams must be appointed within the first two wee...
  Chunk 6 (166 chars): 3. Cultural integration: Cultural differences must be proactively addressed through joint ...
  Chunk 7 (269 chars): ## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) foc...
  Chunk 8 (215 chars): ## Appendix

Detailed case studies from recent financial services and healthcare M&A trans...

Naive fixed-size splitting: 8 chunks
  Chunk 1 (48 chars): # Post-Merger 